# Fake News Detection - 02 Logistic Regression Baseline

In [ ]:
import os
import sys

sys.path.insert(0, '../src')

import polars as pl

from data import standard_labels, binary_labels, label_distribution, split_data
from evaluation import report
from features import bow, count_vectorizer, save_vectorizer
from models import save_model, train_logreg

In [ ]:
df = pl.read_csv('../data/preprocessed/995,000_rows_preprocessed.csv', n_rows=10_000)
df = df.select(['type', 'content'])
df = standard_labels(df)
df.head()

In [ ]:
df_train, df_val, df_test = split_data(df)

print('Train split:')
print(label_distribution(df_train))
print('\nVal split:')
print(label_distribution(df_val))
print('\nTest split:')
print(label_distribution(df_test))

In [ ]:
vectorizer = count_vectorizer(max_features=int(1e6), ngram_range=(1, 3))

X_train = bow(df_train, vectorizer, fit_transform=True)
X_val = bow(df_val, vectorizer, fit_transform=False)
X_test = bow(df_test, vectorizer, fit_transform=False)

y_train = binary_labels(df_train)
y_val = binary_labels(df_val)
y_test = binary_labels(df_test)

In [ ]:
classifier = train_logreg(X_train, y_train, random_state=1000, max_iter=2500)
y_val_pred = classifier.predict(X_val)
summary = report(y_val, y_val_pred)
print(summary)

In [ ]:
save_model(classifier, '../artifacts/models/02_logreg_model.joblib')
save_vectorizer(vectorizer, '../artifacts/vectorizers/02_logreg_vectorizer.joblib')
with open('../artifacts/results/02_logreg_validation_report.txt', 'w', encoding='utf-8') as file:
    file.write(summary)